# Template_Toyco — lendo o relatório de sensibilidade do CPLEX

**Pesquisa Operacional I · Análise de Sensibilidade Básica**

Este notebook é o **template** da Parte 1 do laboratório. O objetivo aqui não é re-modelar a Toyco (Taha §3.6), e sim **aprender a ler o relatório de sensibilidade** do CPLEX em uma estrutura indexada — exatamente a estrutura que você vai usar para o NeuralCloud na Parte 2.

**Fluxo do notebook**

1. Modelo `toyco.mod` (indexado por operação e por produto).
2. Dados `toyco.dat`.
3. Resolver com `cplex_options = "sens=1"`.
4. **Tabela de restrições** — para cada `Capacidade[i]`: $b_i$, folga, $y_i$, `sensrhslo`, `sensrhshi`, `status`.
5. **Tabela de variáveis** — para cada `x[j]`: $c_j$, $x_j^*$, `rc`, `sensobjlo`, `sensobjhi`.
6. Conferir os números com o **Gabarito Rápido** do roteiro.
7. Responder a **Tarefa 1.2** no Markdown, usando a **Regra dos Três Elementos**.

**Atenção.** O código deste notebook já está pronto. O desafio começa em **Parte 2 — NeuralCloud**, onde você vai *copiar o padrão de extração daqui* e adaptar para variáveis duplamente indexadas $x[i,j]$ e várias famílias de restrição.


## 0. Setup

Rode a célula abaixo **uma vez** por sessão. Ela instala o `amplpy`, baixa o módulo do CPLEX e instancia um objeto `ampl` já configurado.

In [1]:
!pip install -q amplpy
from amplpy import AMPL, ampl_notebook
import pandas as pd

ampl = ampl_notebook(
    modules=["cplex"],
    license_uuid="default",
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.7 MB/s eta 0:00:00


## 1. Modelo da Toyco em AMPL (`toyco.mod`)

Para deixar a estrutura **paralela ao NeuralCloud**, escrevemos a Toyco com conjuntos e parâmetros indexados — em vez de três variáveis soltas `x1, x2, x3`, usamos `x[j]` com `j ∈ PROD`. É exatamente o estilo que você usará na Parte 2.

A célula abaixo usa `%%writefile` para gravar o conteúdo em um arquivo `.mod` no servidor. Não é Python — é uma instrução do ambiente Colab que diz "salve esta célula como arquivo".

In [2]:
%%writefile toyco.mod
# ---- Toyco: análise de sensibilidade (Taha §3.6) ----
set OP;       # operações  (Op1, Op2, Op3)
set PROD;     # produtos   (Trem, Caminhao, Carro)

param margem {PROD} >= 0;      # receita por unidade ($)
param tempo  {OP, PROD} >= 0;  # min de operação i por unidade do produto j
param cap    {OP} >= 0;        # capacidade diária de cada operação (min/dia)

var x {PROD} >= 0;             # quantidade produzida de cada produto

maximize z: sum {j in PROD} margem[j] * x[j];

s.t. Capacidade {i in OP}:
    sum {j in PROD} tempo[i,j] * x[j] <= cap[i];

Writing toyco.mod


## 2. Dados (`toyco.dat`)

Receitas $(3, 2, 5)$ para (Trem, Caminhão, Carro). Tempos de operação conforme a tabela do roteiro. Capacidades diárias: 430, 460 e 420 min.

In [3]:
%%writefile toyco.dat
set OP   := Op1 Op2 Op3 ;
set PROD := Trem Caminhao Carro ;

param margem :=
    Trem      3
    Caminhao  2
    Carro     5 ;

param tempo : Trem  Caminhao  Carro :=
    Op1        1      2        1
    Op2        3      0        2
    Op3        1      4        0 ;

param cap :=
    Op1  430
    Op2  460
    Op3  420 ;

Writing toyco.dat


## 3. Resolver com CPLEX e `sens=1`

A opção `cplex_options = "sens=1"` é o que dispara o relatório completo de sensibilidade. Sem ela, os atributos `sensrhslo/hi` e `sensobjlo/hi` ficam vazios.

In [4]:
ampl.reset()
ampl.read("toyco.mod")
ampl.read_data("toyco.dat")

ampl.option["solver"] = "cplex"
ampl.option["cplex_options"] = "sens=1"
ampl.solve()

z = ampl.get_objective("z").value()
print(f"z* = $ {z:,.2f}")

x_star = ampl.get_variable("x").get_values().to_pandas()
x_star.columns = ["x*"]
print("\nAlocação ótima:")
display(x_star)

CPLEX 22.1.2:   alg:sens = 1
CPLEX 22.1.2: optimal solution; objective 1350
3 simplex iterations

suffix up OUT;
suffix down OUT;
suffix current OUT;
suffix sensobj OUT;
suffix senslbhi OUT;
suffix senslblo OUT;
suffix sensubhi OUT;
suffix sensublo OUT;
suffix sensobjhi OUT;
suffix sensobjlo OUT;
suffix sensrhshi OUT;
suffix sensrhslo OUT;
z* = $ 1,350.00

Alocação ótima:


,x*
Caminhao,100
Carro,230
Trem,0


## 4. Tabela de Restrições — preço-sombra e faixa de viabilidade

**Este bloco é o coração do template.** Para cada restrição da família `Capacidade[i]` extraímos:

| coluna | atributo AMPL | significado |
|---|---|---|
| `body` | `Capacidade.body` | LHS calculado no ótimo (consumo real) |
| `b`    | `Capacidade.ub`   | RHS atual ($b_i$) |
| `y`    | `Capacidade.dual` | preço-sombra $y_i$ |
| `rhslo`/`rhshi` | `Capacidade.sensrhslo/hi` | faixa de viabilidade |

A coluna `folga = b - body` e o `status` (ativa/folgada) são calculados em Python.

> **Padrão a memorizar.** `ampl.get_data("Família.body", "Família.ub", "Família.dual", "Família.sensrhslo", "Família.sensrhshi").to_pandas()` devolve um DataFrame indexado pelo conjunto que indexa a família. Para o NeuralCloud, você vai usar esse mesmo padrão para cada família (`Capac`, `Potencia`, `Demanda`, ...) e concatenar os resultados.

In [5]:
df_restr = ampl.get_data(
    "Capacidade.body",
    "Capacidade.ub",
    "Capacidade.dual",
    "Capacidade.sensrhslo",
    "Capacidade.sensrhshi",
).to_pandas()
df_restr.columns = ["body", "b", "y", "rhslo", "rhshi"]
df_restr["folga"]  = df_restr["b"] - df_restr["body"]
df_restr["status"] = df_restr["folga"].apply(
    lambda f: "ativa" if abs(f) < 1e-6 else "folgada"
)
df_restr = df_restr[["b", "folga", "y", "rhslo", "rhshi", "status"]]
df_restr = df_restr.sort_values("y", ascending=False)
display(df_restr.round(2))

,b,folga,y,rhslo,rhshi,status
Op2,460,0,2,440,860,ativa
Op1,430,0,1,230,440,ativa
Op3,420,20,0,400,100000000000000000000,folgada


## 5. Tabela de Variáveis — custo reduzido e faixa de otimalidade

Para cada variável `x[j]` extraímos:

| coluna | atributo AMPL | significado |
|---|---|---|
| `c_j`  | `margem[j]`   | coeficiente atual na f.o. |
| `x*`   | `x.val`       | valor ótimo |
| `rc`   | `x.rc`        | custo reduzido |
| `objlo`/`objhi` | `x.sensobjlo/hi` | faixa de otimalidade |

**Convenção de sinal do `rc`.** Para variáveis com $x_j^*=0$ em um problema de maximização, o CPLEX via AMPL costuma reportar `rc` *negativo*. O módulo desse valor é o quanto $c_j$ precisaria aumentar para a variável valer a pena entrar no plano.

In [6]:
df_var = ampl.get_data(
    "x", "x.rc", "x.sensobjlo", "x.sensobjhi"
).to_pandas()
df_var.columns = ["x*", "rc", "objlo", "objhi"]

margem = ampl.get_parameter("margem").to_pandas()
margem.columns = ["c_j"]

df_var = df_var.join(margem)
df_var = df_var[["c_j", "x*", "rc", "objlo", "objhi"]]
display(df_var.round(2))

,c_j,x*,rc,objlo,objhi
Caminhao,2,100,0,0,10
Carro,5,230,0,2.333333,100000000000000000000
Trem,3,0,-4,-100000000000000000000,7


## 6. Conferindo com o **Gabarito Rápido** do roteiro

Compare as suas duas tabelas com o gabarito da §3.1 do PDF:

**Restrições**

| restr. | $b$ | folga | $y$ | rhslo | rhshi | status |
|--------|-----|-------|-----|-------|-------|--------|
| Op.1   | 430 | 0     | 1   | 230   | 440      | ativa   |
| Op.2   | 460 | 0     | 2   | 440   | 860      | ativa   |
| Op.3   | 420 | 20    | 0   | 400   | $\infty$ | folgada |

**Variáveis**

| var.        | $c_j$ | $x_j^*$ | rc  | objlo     | objhi    |
|-------------|-------|---------|-----|-----------|----------|
| Trem        | 3     | 0       | -4  | $-\infty$ | 7        |
| Caminhão    | 2     | 100     | 0   | 0         | 10       |
| Carro       | 5     | 230     | 0   | 2.33      | $\infty$ |

Se as suas tabelas batem com isso, você está pronto para a Tarefa 1.2.

## 7. Tarefa 1.2 — Decisão Gerencial (responda aqui)

Use **somente** as tabelas acima, sem rodar o modelo de novo. Em cada item, aplique a **Regra dos Três Elementos** do roteiro:

> **(1)** preço-sombra ou `rc` envolvido — **(2)** variação proposta — **(3)** faixa de validade.

Resposta que cita o preço-sombra e *esquece a faixa* será considerada parcial.

---

**(a)** Vale a pena alugar capacidade extra da Op.1 a \$0,80/min?

> *(sua resposta — clique duas vezes para editar)*

**(b)** A gerência sugere subir a Op.2 de 460 para 600 min. Qual o ganho previsto em $z$? E se subir para 900 min?

> *(sua resposta)*

**(c)** Por que aumentar a Op.3 não muda $z$, embora tenha folga de apenas 20 min?

> *(sua resposta)*

## 8. Como adaptar este template para o **NeuralCloud**

A estrutura do NeuralCloud é **paralela** à da Toyco — só com mais dimensões:

|                 | Toyco                          | NeuralCloud                                          |
|-----------------|--------------------------------|------------------------------------------------------|
| Conjuntos       | `OP`, `PROD`                   | `DC`, `PLANO`                                        |
| Variáveis       | `var x {PROD}` (1 índice)      | `var x {DC, PLANO}` (2 índices)                      |
| Restrições      | `Capacidade {OP}` (1 família)  | `Capac {DC}`, `Potencia {DC}`, `Demanda {PLANO}`, ... |

**Roteiro de adaptação.**

1. **Modelo e dados.** O MiniLab já entregou `bruto.mod` e `bruto.dat`. Use `%%writefile` como aqui.
2. **Tabela de restrições.** Para cada família, repita o bloco da Seção 4 deste notebook trocando `"Capacidade"` pelo nome da família. Use `pd.concat([...])` para juntar tudo em uma única tabela e depois ordene por `y` decrescente.
3. **Tabela de variáveis.** O padrão é idêntico ao da Seção 5 — apenas o índice agora tem dois níveis (`DC, PLANO`); o `.to_pandas()` já devolve isso pronto, basta cuidar dos nomes das colunas.
4. **Decisões.** Aplique a **Regra dos Três Elementos** em cada item do roteiro (Tarefas 2.2 e 2.3).

Boa sorte. Este é o ponto da disciplina onde a análise de sensibilidade deixa de ser exercício no papel e vira ferramenta de decisão real.